# Cassava Leaf Disease Classification

Transfer-learning image classifier for the [Kaggle Cassava Leaf Disease Classification](https://www.kaggle.com/c/cassava-leaf-disease-classification) competition.

Given a photo of a cassava leaf, predict one of **5 classes** (4 diseases + healthy). This is a clean, self-contained **PyTorch** pipeline (no external training framework): data loading with `albumentations`, a pretrained **ResNet18** backbone, mixed-precision training, early stopping, and Kaggle-format submission generation.

## 1. Setup

In [ ]:
# Install dependencies (uncomment on Colab / Kaggle if needed)
# !pip install -q torch torchvision albumentations opencv-python-headless pandas scikit-learn tqdm matplotlib

In [ ]:
import os
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn import metrics, model_selection

%matplotlib inline

## 2. Configuration

`DATA_DIR` is **auto-detected** for Kaggle, Colab, and local runs by looking for the folder that contains `train.csv`. If detection picks the wrong place, set `CFG.DATA_DIR` manually.

In [ ]:
import glob


def find_data_dir():
    """Locate the folder that actually contains train.csv.

    Checks the usual Kaggle/Colab/local spots, then falls back to searching
    recursively under /kaggle/input and /content, so it works no matter what
    the attached dataset folder is named.
    """
    exact = [
        "/kaggle/input/cassava-leaf-disease-classification",   # Kaggle competition
        "/content/cassava-leaf-disease-classification",        # Colab subfolder
        "/content",                                            # Colab /content
        ".",                                                   # local / current dir
    ]
    for c in exact:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c

    for root in ["/kaggle/input", "/content", "."]:
        if os.path.isdir(root):
            hits = glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
            if hits:
                return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Could not find train.csv. Set CFG.DATA_DIR manually to the folder "
        "containing train.csv, train_images/ and test_images/."
    )


class CFG:
    DATA_DIR    = find_data_dir()             # auto-detected; override manually if needed
    TRAIN_CSV   = os.path.join(DATA_DIR, "train.csv")
    TRAIN_DIR   = os.path.join(DATA_DIR, "train_images")
    TEST_DIR    = os.path.join(DATA_DIR, "test_images")
    SUB_CSV     = os.path.join(DATA_DIR, "sample_submission.csv")

    IMG_SIZE    = 256
    NUM_CLASSES = 5
    BATCH_SIZE  = 32
    EPOCHS      = 5
    LR          = 1e-3
    VALID_SIZE  = 0.1
    PATIENCE    = 2                           # early-stopping patience (epochs)
    NUM_WORKERS = 2
    SEED        = 42
    MODEL_PATH  = "model.bin"

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Data dir :", CFG.DATA_DIR)
print("Device   :", CFG.DEVICE)
assert os.path.isdir(CFG.TRAIN_DIR), f"train_images/ not found next to train.csv in {CFG.DATA_DIR}"

In [ ]:
def seed_everything(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

## 3. Load the labels

The dataset is **imbalanced** — class 3 (CMD) dominates — which is worth keeping in mind when reading accuracy.

In [ ]:
LABEL_NAMES = {
    0: "Cassava Bacterial Blight (CBB)",
    1: "Cassava Brown Streak Disease (CBSD)",
    2: "Cassava Green Mottle (CGM)",
    3: "Cassava Mosaic Disease (CMD)",
    4: "Healthy",
}

df = pd.read_csv(CFG.TRAIN_CSV)
print(df.shape)
df.head()

In [ ]:
df["label"].value_counts().sort_index()

In [ ]:
# Hold-out validation split (use StratifiedKFold for a stronger setup)
df_train, df_valid = model_selection.train_test_split(
    df,
    test_size=CFG.VALID_SIZE,
    random_state=CFG.SEED,
    stratify=df["label"],
)
df_train = df_train.reset_index(drop=True)
df_valid = df_valid.reset_index(drop=True)
print("train:", df_train.shape, "valid:", df_valid.shape)

## 4. Dataset and augmentations

Images are read with OpenCV (BGR → RGB) and transformed with `albumentations`. Training uses random crops/flips/rotations; validation and test use a deterministic center crop. Both normalize with ImageNet statistics.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_aug = A.Compose([
    A.RandomResizedCrop(height=CFG.IMG_SIZE, width=CFG.IMG_SIZE, scale=(0.8, 1.0)),
    A.Transpose(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=255.0),
    ToTensorV2(),
])

valid_aug = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.CenterCrop(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=255.0),
    ToTensorV2(),
])

In [ ]:
class CassavaDataset(Dataset):
    def __init__(self, image_ids, image_dir, targets=None, augmentations=None):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.targets = targets
        self.augmentations = augmentations

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_ids[idx])
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.augmentations is not None:
            image = self.augmentations(image=image)["image"]

        if self.targets is not None:
            target = torch.tensor(self.targets[idx], dtype=torch.long)
            return image, target
        return image

In [ ]:
train_dataset = CassavaDataset(
    image_ids=df_train["image_id"].values,
    image_dir=CFG.TRAIN_DIR,
    targets=df_train["label"].values,
    augmentations=train_aug,
)
valid_dataset = CassavaDataset(
    image_ids=df_valid["image_id"].values,
    image_dir=CFG.TRAIN_DIR,
    targets=df_valid["label"].values,
    augmentations=valid_aug,
)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

In [ ]:
# Peek at one augmented training image
def show_sample(dataset, idx=0):
    image, target = dataset[idx]
    img = image.permute(1, 2, 0).numpy()
    img = (img * IMAGENET_STD) + IMAGENET_MEAN   # un-normalize for display
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(f"label {int(target)}: {LABEL_NAMES[int(target)]}")
    plt.axis("off")
    plt.show()

show_sample(train_dataset, 0)

## 5. Model

`torchvision` **ResNet18** pretrained on ImageNet, with the final fully-connected layer replaced for the 5-class problem. The helper handles both the new (`weights=`) and old (`pretrained=`) torchvision APIs.

In [ ]:
def build_model(num_classes=CFG.NUM_CLASSES, pretrained=True):
    try:
        from torchvision.models import ResNet18_Weights
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = torchvision.models.resnet18(weights=weights)
    except (ImportError, AttributeError):
        model = torchvision.models.resnet18(pretrained=pretrained)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_model().to(CFG.DEVICE)

## 6. Training and evaluation

`CrossEntropyLoss`, Adam, a StepLR schedule, and automatic mixed precision (`torch.cuda.amp`) when a GPU is available.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, targets)
        running_loss += loss.item() * images.size(0)
        all_preds.append(outputs.argmax(1).cpu().numpy())
        all_targets.append(targets.cpu().numpy())
    preds = np.concatenate(all_preds)
    tgts = np.concatenate(all_targets)
    acc = metrics.accuracy_score(tgts, preds)
    return running_loss / len(loader.dataset), acc

## 7. Training loop

Trains for up to `EPOCHS`, saving the best weights by validation accuracy and stopping early if it does not improve for `PATIENCE` epochs.

In [ ]:
best_acc = 0.0
epochs_no_improve = 0

for epoch in range(1, CFG.EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, CFG.DEVICE)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, CFG.DEVICE)
    scheduler.step()

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | "
          f"valid_loss {valid_loss:.4f} | valid_acc {valid_acc:.4f}")

    if valid_acc > best_acc:
        best_acc = valid_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), CFG.MODEL_PATH)
        print(f"  -> saved new best model (valid_acc {best_acc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= CFG.PATIENCE:
            print(f"Early stopping at epoch {epoch} (best valid_acc {best_acc:.4f})")
            break

print(f"Best validation accuracy: {best_acc:.4f}")

## 8. Inference and submission

Load the best checkpoint, predict on the test images, and write `submission.csv` in the Kaggle format (`image_id,label`).

In [ ]:
test_df = pd.read_csv(CFG.SUB_CSV)

test_dataset = CassavaDataset(
    image_ids=test_df["image_id"].values,
    image_dir=CFG.TEST_DIR,
    targets=None,
    augmentations=valid_aug,
)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=CFG.DEVICE))
model.eval()

all_preds = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(CFG.DEVICE)
        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            outputs = model(images)
        all_preds.append(outputs.argmax(1).cpu().numpy())

test_df["label"] = np.concatenate(all_preds)
test_df.to_csv("submission.csv", index=False)
test_df.head()

## Notes / possible improvements

- **Stronger backbone** via `timm` (EfficientNet / ResNeXt) instead of ResNet18.
- **K-fold cross-validation** and ensembling rather than a single hold-out split.
- **Class-imbalance handling** (class-weighted loss or a weighted sampler) given the CMD skew.
- **Test-time augmentation (TTA)** and a cosine LR schedule with warm-up.
- **Label smoothing**, which tends to help on this noisy-label dataset.